In [2]:
from pathlib import Path
import sys
import math
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from typing import Optional, Tuple, Union


from src.perturbation_methods.perturbation_methods import *

In [4]:
from pathlib import Path
import sys, os, time
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader.data_loader import PhiSatSegDataModule, Sentinel2SegDataModule
from src.perturbation_methods.perturbation_methods import *

In [6]:
# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

DATASET_ROOT = Path("/shared/home/ivanderspoel/scratch/segmentation_dataset_v1")

PHISAT2_MODEL_PATH = Path(
    "/shared/home/ivanderspoel/msc_thesis/results/"
    "generation_10_phisat2_clean/gen_10_id_11404771159034136610.pt"
)

S2_MODEL_PATH = Path(
    "/shared/home/ivanderspoel/msc_thesis/results/"
    "generation_13_sentinel2/gen_13_id_5392313798277613390.pt"
)

BS = 32
NW = 2
SEED = 42

pl.seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("medium")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Seed set to 42


device(type='cuda')

In [7]:
# -----------------------------------------------------------------------------
# Datamodules like main.py
# -----------------------------------------------------------------------------

phisat2_dm = PhiSatSegDataModule(
    image_dir=str(DATASET_ROOT / "images_phisat2_npy"),
    mask_dir=str(DATASET_ROOT / "masks_phisat2_npy"),
    batch_size=BS,
    val_split=0.2,
    num_workers=NW,
    perturbation="clean",
    strength=0.0,
)

s2_dm = Sentinel2SegDataModule(
    image_dir=str(DATASET_ROOT / "images_s2_npy"),
    mask_dir=str(DATASET_ROOT / "masks_s2_npy"),
    batch_size=BS,
    val_split=0.2,
    num_workers=NW,
    perturbation="clean",
    strength=0.0,
)

phisat2_dm.setup(stage="fit")
phisat2_dm.setup(stage="test")

s2_dm.setup(stage="fit")
s2_dm.setup(stage="test")

print("PhiSat-2 input shape:", phisat2_dm.input_shape)
print("Sentinel-2 input shape:", s2_dm.input_shape)

PhiSat-2 input shape: torch.Size([8, 256, 256])
Sentinel-2 input shape: torch.Size([7, 128, 128])


In [8]:
# -----------------------------------------------------------------------------
# Load TorchScript models directly
# -----------------------------------------------------------------------------

phisat2_model = torch.jit.load(str(PHISAT2_MODEL_PATH), map_location=device)
phisat2_model.eval().to(device)

s2_model = torch.jit.load(str(S2_MODEL_PATH), map_location=device)
s2_model.eval().to(device)

print("Loaded PhiSat-2:", PHISAT2_MODEL_PATH)
print("Loaded Sentinel-2:", S2_MODEL_PATH)

Loaded PhiSat-2: /shared/home/ivanderspoel/msc_thesis/results/generation_10_phisat2_clean/gen_10_id_11404771159034136610.pt
Loaded Sentinel-2: /shared/home/ivanderspoel/msc_thesis/results/generation_13_sentinel2/gen_13_id_5392313798277613390.pt


In [9]:
# -----------------------------------------------------------------------------
# Metrics
# -----------------------------------------------------------------------------

def _update_confmat(preds, targets, num_classes):
    valid = (targets >= 0) & (targets < num_classes)

    inds = num_classes * targets[valid] + preds[valid]

    return torch.bincount(
        inds,
        minlength=num_classes * num_classes,
    ).reshape(num_classes, num_classes)


def _miou_from_confmat(confmat):
    confmat = confmat.float()

    intersection = torch.diag(confmat)
    union = confmat.sum(dim=1) + confmat.sum(dim=0) - intersection

    valid = union > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=confmat.device)

    return (intersection[valid] / union[valid]).mean()


def mask_to_indices(y):
    """
    Supports one-hot masks [B,C,H,W] or index masks [B,H,W].
    """
    if y.ndim == 4:
        return torch.argmax(y, dim=1)
    return y.long()

In [10]:
# -----------------------------------------------------------------------------
# Perturbations
# -----------------------------------------------------------------------------
# Noise:
#   snr_factor=0.25 matched REOBench sigma≈0.08 post-normalization in your sweep.
#
# Blur:
#   PhiSat-2: 0.039 = strongest realistic PhiSat-2 MTF@Nyquist.
#   Sentinel-2: 0.15 = strong realistic Sentinel-2 10 m MTF value.
#
# The perturbation functions are the same sensor-aware ones from your perturbation file.
# Lower SNR factor -> stronger noise. Lower MTF@Nyquist -> stronger blur.
# -----------------------------------------------------------------------------

PHISAT2_PERTURBATIONS = {
    "clean": ("clean", None),
    "misalignment_2px": ("misalignment", 2.0),
    "snr_0.25": ("noise", 0.25),
    "haze_t0.5": ("haze", 0.5),
    "brightness_1.4": ("brightness", 1.4),
    "blur_mtf_0.039": ("blur", 0.039),
}

S2_PERTURBATIONS = {
    "clean": ("clean", None),
    "misalignment_2px": ("misalignment", 2.0),
    "snr_0.25": ("noise", 0.25),
    "haze_t0.5": ("haze", 0.5),
    "brightness_1.4": ("brightness", 1.4),
    "blur_mtf_0.15": ("blur", 0.15),
}

In [11]:
def perturb_batch_raw(
    x,
    perturbation,
    strength,
    seed=0,
    use_official_lref_for_noise=False,
):
    """
    x: raw unnormalized tensor, shape [B,C,H,W].

    Applies perturbation in raw space, then evaluation normalizes afterwards.
    This mirrors Population.evaluate_individual's raw -> perturb -> normalize flow.
    """
    if perturbation in [None, "none", "clean"]:
        return x.clone()

    device = x.device
    dtype = x.dtype

    x_np = x.detach().cpu().numpy().astype(np.float32)
    x_out = np.empty_like(x_np, dtype=np.float32)

    k_grid = None
    if perturbation == "blur":
        h, w = x_np.shape[-2:]
        k_grid = precompute_k_grid(h, w)

    for i in range(x_np.shape[0]):
        xi = x_np[i]
        sample_seed = None if seed is None else int(seed + i)

        if perturbation == "noise":
            xi_out = perturb_snr(
                xi,
                snr_factor=float(strength),
                seed=sample_seed,
                use_official_lref=use_official_lref_for_noise,
            )

        elif perturbation == "blur":
            xi_out = perturb_mtf(
                xi,
                mtf_nyquist=float(strength),
                k_grid=k_grid,
            )

        elif perturbation == "misalignment":
            xi_out = perturb_band_misalignment(
                xi,
                max_shift_px=float(strength),
                reference_band="red",
                seed=sample_seed,
            )

        elif perturbation == "haze":
            xi_out = perturb_haze(
                xi,
                t=float(strength),
                atmospheric_light="p95",
            )

        elif perturbation == "brightness":
            xi_out = perturb_brightness(
                xi,
                alpha=float(strength),
            )

        else:
            raise ValueError(f"Unknown perturbation: {perturbation}")

        x_out[i] = xi_out.astype(np.float32)

    return torch.from_numpy(x_out).to(device=device, dtype=dtype)

In [12]:
# -----------------------------------------------------------------------------
# Evaluation on TorchScript model
# -----------------------------------------------------------------------------

def evaluate_torchscript_model(
    model,
    dm,
    perturbation_name,
    perturbation,
    strength,
    num_classes=4,
    seed=42,
    max_batches=None,
    use_official_lref_for_noise=False,
):
    model.eval().to(device)

    dm.setup(stage="test")
    loader = dm.test_dataloader()

    clean_confmat = torch.zeros(
        num_classes, num_classes, device=device, dtype=torch.long
    )
    pert_confmat = torch.zeros(
        num_classes, num_classes, device=device, dtype=torch.long
    )

    consistency_correct = torch.zeros((), device=device, dtype=torch.long)
    consistency_total = torch.zeros((), device=device, dtype=torch.long)

    process_time = 0.0
    n_images = 0

    with torch.inference_mode():
        for batch_idx, batch in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            x, y = batch
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_indices = mask_to_indices(y)

            # raw -> perturb
            x_pert = perturb_batch_raw(
                x,
                perturbation=perturbation,
                strength=strength,
                seed=seed + batch_idx * x.shape[0],
                use_official_lref_for_noise=use_official_lref_for_noise,
            )

            # normalize after perturbation
            x_norm = dm.test_dataset.normalize_batch(x)
            x_pert_norm = dm.test_dataset.normalize_batch(x_pert)

            x_all = torch.cat([x_norm, x_pert_norm], dim=0)

            if device.type == "cuda":
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)

                start.record()
                logits_all = model(x_all)
                end.record()

                torch.cuda.synchronize()
                process_time += start.elapsed_time(end) / 1000.0
            else:
                t0 = time.time()
                logits_all = model(x_all)
                process_time += time.time() - t0

            logits_clean, logits_pert = torch.chunk(logits_all, chunks=2, dim=0)

            preds_clean = torch.argmax(logits_clean, dim=1)
            preds_pert = torch.argmax(logits_pert, dim=1)

            clean_confmat += _update_confmat(preds_clean, y_indices, num_classes)
            pert_confmat += _update_confmat(preds_pert, y_indices, num_classes)

            consistency_correct += (preds_clean == preds_pert).sum()
            consistency_total += preds_clean.numel()

            n_images += x.shape[0]

    miou_clean = _miou_from_confmat(clean_confmat)
    miou_perturbed = _miou_from_confmat(pert_confmat)
    prediction_consistency = consistency_correct.float() / consistency_total.float()

    return {
        "perturbation_name": perturbation_name,
        "perturbation": perturbation,
        "strength": strength,
        "miou_clean": float(miou_clean.detach().cpu()),
        "miou_perturbed": float(miou_perturbed.detach().cpu()),
        "miou_drop": float((miou_clean - miou_perturbed).detach().cpu()),
        "prediction_consistency": float(prediction_consistency.detach().cpu()),
        "process_time_sec": float(process_time),
        "n_images": int(n_images),
    }


def evaluate_suite(
    model,
    dm,
    suite,
    dataset_name,
    max_batches=None,
    use_official_lref_for_noise=False,
):
    rows = []

    for name, (perturbation, strength) in suite.items():
        print(f"[{dataset_name}] {name}")

        row = evaluate_torchscript_model(
            model=model,
            dm=dm,
            perturbation_name=name,
            perturbation=perturbation,
            strength=strength,
            num_classes=4,
            seed=SEED,
            max_batches=max_batches,
            use_official_lref_for_noise=use_official_lref_for_noise,
        )

        row["dataset"] = dataset_name
        rows.append(row)

    return pd.DataFrame(rows)

In [13]:
# -----------------------------------------------------------------------------
# Run full evaluation
# -----------------------------------------------------------------------------
# For exact Population.perturb_batch behavior, set use_official_lref_for_noise=True.
# For your calibrated snr_factor=0.25 behavior, keep it False.

phisat2_results = evaluate_suite(
    model=phisat2_model,
    dm=phisat2_dm,
    suite=PHISAT2_PERTURBATIONS,
    dataset_name="phisat2",
    max_batches=None,
    use_official_lref_for_noise=False,
)

s2_results = evaluate_suite(
    model=s2_model,
    dm=s2_dm,
    suite=S2_PERTURBATIONS,
    dataset_name="sentinel2",
    max_batches=None,
    use_official_lref_for_noise=False,
)

results_df = pd.concat([phisat2_results, s2_results], ignore_index=True)

cols = [
    "dataset",
    "perturbation_name",
    "strength",
    "miou_clean",
    "miou_perturbed",
    "miou_drop",
    "prediction_consistency",
    "process_time_sec",
    "n_images",
]

results_df[cols].sort_values(["dataset", "miou_drop"], ascending=[True, False])

[phisat2] clean
[phisat2] misalignment_2px
[phisat2] snr_0.25
[phisat2] haze_t0.5
[phisat2] brightness_1.4
[phisat2] blur_mtf_0.039
[sentinel2] clean
[sentinel2] misalignment_2px
[sentinel2] snr_0.25
[sentinel2] haze_t0.5
[sentinel2] brightness_1.4
[sentinel2] blur_mtf_0.15


,dataset,perturbation_name,strength,miou_clean,miou_perturbed,miou_drop,prediction_consistency,process_time_sec,n_images
3,phisat2,haze_t0.5,0.500,0.589728,0.279147,0.310580,0.522246,45.068553,10000
2,phisat2,snr_0.25,0.250,0.589728,0.470070,0.119658,0.748659,41.432571,10000
5,phisat2,blur_mtf_0.039,0.039,0.589728,0.535124,0.054604,0.829924,39.343321,10000
4,phisat2,brightness_1.4,1.400,0.589728,0.568111,0.021617,0.912601,25.012081,10000
1,phisat2,misalignment_2px,2.000,0.589728,0.583770,0.005957,0.941603,46.154635,10000
0,phisat2,clean,NaN,0.589728,0.589728,0.000000,1.000000,41.113777,10000
9,sentinel2,haze_t0.5,0.500,0.622782,0.224998,0.397784,0.467364,7.161759,10000
8,sentinel2,snr_0.25,0.250,0.622782,0.378079,0.244703,0.603916,9.631171,10000
10,sentinel2,brightness_1.4,1.400,0.622782,0.469057,0.153725,0.684429,6.686601,10000
11,sentinel2,blur_mtf_0.15,0.150,0.622782,0.522797,0.099985,0.782922,5.086624,10000
